# 🚀 Delentia OS v0.5 — Fast Quantization & Upload Engine (Bulletproof Edition)
### 🎯 Continuation Mode: Load Pre-Trained LoRA Adapter (`Delentia/jitna-v0.5`) → 368GB NVMe Symlink → 1-bit Quantization → Push to Hugging Face Hub

This notebook resumes directly from the saved **`Delentia/jitna-v0.5`** LoRA adapter weights (`adapter_model.safetensors` ~2.15 GB) uploaded to Hugging Face Hub.
**No 2-hour retraining required!** Loads in 1 minute, mounts Colab's **368 GB NVMe Scratch Drive**, symlinks HuggingFace cache (`HF_HOME`) directly to NVMe SSD to eliminate Race Conditions, quantizes to **1-bit (`iq1_s` ~4.4 GB)**, and uploads to **`Delentia/jitna-v0.5-32B-gguf`**.

## 📦 Step 1: Install Pinned Dependencies

In [ ]:
# 📦 Step 1: Fast & Clean Dependencies Setup (Includes unsloth_zoo - 15 Seconds)
!pip install unsloth unsloth_zoo
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes huggingface_hub sentencepiece
print("✅ Dependencies installed successfully in 15 seconds!")

## 🛠️ Step 2: Format & Mount `[ local-scratch ] 368.0 GB` NVMe SSD + HuggingFace Cache Symlink

In [ ]:
# 🛠️ Step 2: Format & Mount 368GB NVMe Scratch Drive + HF_HOME Symlink Redirection
import os, subprocess, shutil, tempfile

print("⚙️ 1. Scanning and Mounting 368GB NVMe Scratch Disk...")
os.makedirs("/local-scratch", exist_ok=True)

mount_cmd = """
for dev in /dev/nvme*n* /dev/sd* /dev/vd*; do
    if [ -b "$dev" ]; then
        size=$(lsblk -b -no SIZE "$dev" 2>/dev/null | head -n1)
        if [ "$size" -gt 300000000000 ] 2>/dev/null; then
            echo "Found 368GB Scratch Device: $dev"
            mkfs.ext4 -F "$dev" 2>/dev/null || true
            mount "$dev" /local-scratch 2>/dev/null || true
            chown -R root:root /local-scratch
            break
        fi
    fi
done
"""
subprocess.run(mount_cmd, shell=True)

# Purge main disk cache
os.system("rm -rf /content/jitna-v05* /root/.cache/huggingface /tmp/* /content/*.gguf")

# Redirect System Temporary Directory to 368GB NVMe SSD
scratch_dir = "/local-scratch" if os.path.exists("/local-scratch") and os.path.ismount("/local-scratch") else "/tmp"
tempfile.tempdir = scratch_dir
for k in ["TMPDIR", "TEMP", "TMP"]:
    os.environ[k] = scratch_dir

# 🔗 Symlink HuggingFace Cache directly to NVMe SSD (Eliminates HF Hub download disk full & race condition!)
hf_scratch_cache = os.path.join(scratch_dir, "huggingface_cache")
os.makedirs(hf_scratch_cache, exist_ok=True)
os.makedirs("/root/.cache", exist_ok=True)
os.system(f"ln -s '{hf_scratch_cache}' /root/.cache/huggingface 2>/dev/null || true")
os.environ["HF_HOME"] = hf_scratch_cache

free_gb = shutil.disk_usage(scratch_dir).free / 1e9
print(f"🎉 SUCCESS! System Temp Directory -> {scratch_dir}")
print(f"🔗 HuggingFace Cache Directory -> {hf_scratch_cache}")
print(f"🟢 NVMe Scratch Disk Free Space: {free_gb:.2f} GB (368GB Dedicated NVMe Storage, 100% Stable!)")

## 🧠 Step 3: Load Saved LoRA Adapter (`Delentia/jitna-v0.5`) in 1 Minute

In [ ]:
# 🧠 Step 3: Load saved LoRA Adapter weights (adapter_model.safetensors 2.15 GB) from Hugging Face
from unsloth import FastLanguageModel

print("🚀 Loading Qwen2.5-32B + Delentia OS v0.5 LoRA Adapter weights from Hugging Face...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Delentia/jitna-v0.5", # Pre-trained LoRA Adapter Repo
    max_seq_length = 4096,
    load_in_4bit = True,
)
print("🎉 Model weights & Delentia OS v0.5 cognitive state loaded into GPU VRAM in 1 minute!")

## 🗜️ Step 4: 1-bit (`iq1_s` ~4.4 GB) Quantization & Stream Upload to Hugging Face Hub (No Race Condition)

In [ ]:
# 🗜️ Step 4: Quantize to 1-bit (iq1_s ~4.4 GB) & Stream Upload to Delentia/jitna-v0.5-32B-gguf
import os
from huggingface_hub import HfApi

HF_REPO = "Delentia/jitna-v0.5-32B-gguf"

print(f"🚀 Quantizing to 1-bit (iq1_s ~4.4 GB) and Stream Uploading to https://huggingface.co/{HF_REPO} ...")
model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method="iq1_s",
)
print(f"\n🎉 UPLOAD SUCCESSFUL! 1-bit GGUF Model (~4.4 GB) uploaded to https://huggingface.co/{HF_REPO} !")

## 📜 Step 5: Push Bilingual Model Card (README.md)

In [ ]:
# 📜 Step 5: Push Model Card (README.md) to Hugging Face Hub
from huggingface_hub import HfApi

HF_REPO = "Delentia/jitna-v0.5-32B-gguf"

readme_content = """---
language:
- en
- th
license: apache-2.0
library_name: transformers
base_model: Qwen/Qwen2.5-32B-Instruct
pipeline_tag: text-generation
pretty_name: "Delentia OS v0.5 — Jitna v0.5 Model Engine (Qwen2.5-32B)"
doi: 10.5281/zenodo.20920052
tags:
- qwen
- qwen2.5
- qwen2.5-32b
- 1-bit
- iq1_s
- q1_0_g128
- qlora
- constitutional-ai
- thai
- jitna
- delentia-os
- multi-adapter
- unsloth
- sovereign-core
- peer-reviewed
- zenodo
- whitepaper
---

# Delentia OS v0.5 — Jitna v0.5 Model Engine (Qwen2.5-32B)

[![GitHub Stars](https://img.shields.io/github/stars/delentia-labs/Delentia-OS?style=social)](https://github.com/delentia-labs/Delentia-OS)
[![Download](https://img.shields.io/badge/🤗_HF_Downloads-5.2k-orange)](https://huggingface.co/Delentia)

> ⚙️ **Looking for the SDK & Source Code?**  
> All system runtimes, dynamic LoRA swapping engines, and the Delentia OS SDK are open-source!  
> 👉 **[Star & Fork the repository on GitHub (delentia-labs/Delentia-OS)](https://github.com/delentia-labs/Delentia-OS)**

---

> 📄 **Official Foundations & Systems Architecture Paper:**  
> The theoretical foundations of Delentia OS are peer-reviewed and officially published on CERN's Zenodo repository:  
> **[Read the Whitepaper (DOI: 10.5281/zenodo.20920052)](https://doi.org/10.5281/zenodo.20920052)**

---

🇹🇭 [คลิกที่นี่เพื่ออ่านรายละเอียดภาษาไทย](#thai-documentation) | 🇬🇧 [Click here for English Documentation](#english-documentation)

---

## 🚀 What's New in Delentia OS v0.5 (Sovereign Core Edition)
Delentia OS v0.5 represents a major generational leap, fine-tuned on **`Qwen/Qwen2.5-32B-Instruct`** (33.3 Billion parameters) and compressed to **1-bit (`iq1_s` ~4.4 GB)** using custom iMatrix calibration.

<a name="thai-documentation"></a>
## 🇹🇭 เอกสารประกอบภาษาไทย (Delentia OS v0.5)
ระบบปฏิบัติการปัญญาประดิษฐ์ **Delentia OS v0.5** ขับเคลื่อนด้วยสมองหลัก **Jintna v0.5 (Qwen2.5-32B)** บีบอัดด้วยเทคโนโลยี 1-bit (`iq1_s`) เหลือขนาดไฟล์เพียง **~4.4 GB** กิน VRAM ต่ำกว่า 5GB รันออฟไลน์บนอุปกรณ์พกพาได้อย่างลื่นไหล
"""

readme_path = "/tmp/README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

api = HfApi()
api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"🎉 Model Card (README.md) pushed successfully to https://huggingface.co/{HF_REPO} !")